# 01 — Vetores e Produto Interno: a Geometria dos Embeddings

**Módulo:** `00_foundations/math_linear_algebra`  
**Pré-requisitos:** nenhum  
**Tempo estimado:** 2-3 horas  

---

## Motivação

Um token em um LLM é representado como um vetor em $\mathbb{R}^d$, onde $d$ é a dimensão do embedding (tipicamente 768 a 4096). Toda a operação de atenção é, fundamentalmente, cálculo de similaridade entre vetores via produto interno. Sem entender vetores, a mecânica de attention é opaca.

**Objetivo deste notebook:** implementar as operações vetoriais fundamentais sem nenhuma biblioteca, derivar por que o produto interno é uma medida de similaridade, e conectar esse resultado com a operação `QKᵀ` no mecanismo de atenção.

## 1. Representação de Vetores

Um vetor $\mathbf{v} \in \mathbb{R}^n$ é uma sequência ordenada de $n$ números reais:

$$\mathbf{v} = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_n \end{bmatrix}$$

No contexto de LLMs, cada dimensão de um embedding representa uma característica latente aprendida durante o treinamento.

In [ ]:
# Python puro — sem NumPy, sem nenhuma biblioteca
from typing import List


Vector = List[float]


def dot_product(a: Vector, b: Vector) -> float:
    """
    Produto interno (dot product) entre dois vetores.

    Definição: <a, b> = sum(a_i * b_i) para i = 1..n

    Esta é a operação central do mecanismo de atenção:
    score(q, k) = q · k

    Args:
        a: vetor query (ou qualquer vetor de entrada)
        b: vetor key (ou qualquer vetor de comparação)

    Returns:
        escalar que mede a projeção de a sobre b

    Raises:
        ValueError: se os vetores tiverem dimensões diferentes
    """
    if len(a) != len(b):
        raise ValueError(
            f"Dimensões incompatíveis: {len(a)} != {len(b)}"
        )
    return sum(ai * bi for ai, bi in zip(a, b))


def norm(v: Vector, p: int = 2) -> float:
    """
    Norma Lp de um vetor.

    L2 (euclidiana): ||v||_2 = sqrt(sum(v_i^2))
    L1 (Manhattan): ||v||_1 = sum(|v_i|)

    Em LLMs, a normalização L2 aparece em:
    - Layer Normalization
    - Normalização de embeddings para busca por similaridade
    """
    if p == 2:
        return sum(vi ** 2 for vi in v) ** 0.5
    elif p == 1:
        return sum(abs(vi) for vi in v)
    else:
        raise ValueError(f"Norma L{p} não implementada")


def cosine_similarity(a: Vector, b: Vector) -> float:
    """
    Similaridade do cosseno entre dois vetores.

    cos(a, b) = <a, b> / (||a||_2 * ||b||_2)

    Resultado em [-1, 1]:
      +1: vetores idênticos em direção
       0: vetores ortogonais (sem relação)
      -1: vetores opostos

    Nos embeddings de LLMs, palavras semanticamente
    similares tendem a ter cosine similarity alta.
    """
    numerator = dot_product(a, b)
    denominator = norm(a) * norm(b)
    if denominator == 0:
        raise ValueError("Vetor zero não tem direção definida")
    return numerator / denominator

## 2. Experimento: Similaridade Semântica

Vamos simular o comportamento de embeddings com vetores manuais para entender a geometria antes de usar embeddings reais.

In [ ]:
# Embeddings simulados (em 3D para visualização)
# Na prática, embeddings têm 768-4096 dimensões

rei = [0.9, 0.1, 0.8]     # masculino=alto, nobre=alto
rainha = [0.1, 0.9, 0.8]  # feminino=alto, nobre=alto
homem = [0.9, 0.1, 0.1]   # masculino=alto, nobre=baixo
mulher = [0.1, 0.9, 0.1]  # feminino=alto, nobre=baixo
pedra = [0.0, 0.0, 0.0]   # nenhuma dessas características

pares = [
    ("rei", "rainha", rei, rainha),
    ("rei", "homem", rei, homem),
    ("rainha", "mulher", rainha, mulher),
    ("rei", "pedra", rei, pedra),
    ("homem", "mulher", homem, mulher),
]

print("Similaridade do Cosseno entre pares:\n")
print(f"{'Par':<25} {'Cosine Similarity':>18}")
print("-" * 44)
for nome_a, nome_b, va, vb in pares:
    try:
        sim = cosine_similarity(va, vb)
        print(f"{nome_a} × {nome_b:<20} {sim:>18.4f}")
    except ValueError as e:
        print(f"{nome_a} × {nome_b:<20} {'indefinida (vetor zero)':>18}")

## 3. Conexão com o Mecanismo de Atenção

A operação central do Self-Attention é:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

O produto $QK^\top$ é exatamente o cálculo do produto interno entre **todos os pares** de query e key. Cada elemento $(i,j)$ da matriz resultante mede o quanto o token $i$ deve prestar atenção ao token $j$.

A divisão por $\sqrt{d_k}$ é a normalização: sem ela, para dimensões altas ($d_k$ grande), os produtos internos crescem na escala de $\sqrt{d_k}$, empurrando o softmax para regiões de gradiente extremamente pequeno.

**Exercício:** implemente a função de atenção escalar abaixo usando apenas as funções que você acabou de criar.

In [ ]:
import math


def softmax(scores: Vector) -> Vector:
    """Softmax numericamente estável (subtrai o máximo para evitar overflow)."""
    max_score = max(scores)
    exp_scores = [math.exp(s - max_score) for s in scores]
    total = sum(exp_scores)
    return [e / total for e in exp_scores]


def scaled_dot_product_attention(
    query: Vector,
    keys: List[Vector],
    values: List[Vector],
) -> Vector:
    """
    Atenção escalar (single-head, single query).

    Eq. 1 do paper 'Attention Is All You Need' (Vaswani et al., 2017)
    para um único vetor query e uma sequência de keys/values.

    Args:
        query: vetor de consulta (d_k,)
        keys: lista de K vetores key, cada um (d_k,)
        values: lista de K vetores value, cada um (d_v,)

    Returns:
        vetor de contexto ponderado (d_v,)
    """
    d_k = len(query)

    # Passo 1: scores de atenção não normalizados
    raw_scores = [dot_product(query, k) / math.sqrt(d_k) for k in keys]

    # Passo 2: normalização via softmax → pesos de atenção
    attention_weights = softmax(raw_scores)

    # Passo 3: soma ponderada dos values
    d_v = len(values[0])
    output = [0.0] * d_v
    for weight, value in zip(attention_weights, values):
        for i in range(d_v):
            output[i] += weight * value[i]

    return output


# Teste mínimo
q = [1.0, 0.0, 0.0]
k = [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
v = [[10.0], [20.0], [30.0]]

resultado = scaled_dot_product_attention(q, k, v)
weights_esperados = softmax([1.0 / math.sqrt(3), 0.0, 0.0])

print("Resultado da atenção:", resultado)
print("Pesos esperados (query alinhado com key[0]):", weights_esperados)
print("O valor de saída deve ser dominado por v[0] = 10")

## 4. Verificação com NumPy

Agora que entendemos a mecânica, verificamos nossa implementação contra NumPy.

In [ ]:
import numpy as np

# Replica o experimento acima com NumPy
q_np = np.array([1.0, 0.0, 0.0])
k_np = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
v_np = np.array([[10.0], [20.0], [30.0]])

scores_np = (q_np @ k_np.T) / np.sqrt(len(q_np))
weights_np = np.exp(scores_np - scores_np.max())
weights_np = weights_np / weights_np.sum()
output_np = weights_np @ v_np

print("NumPy output:", output_np)
print("Nossa implementação:", resultado)

diferenca = abs(output_np[0] - resultado[0])
print(f"Diferença: {diferenca:.2e} (deve ser < 1e-10)")
assert diferenca < 1e-10, "Implementação diverge do NumPy!"
print("Verificação OK.")

## Resumo

Neste notebook derivamos e implementamos:

- **Produto interno** como operação de similaridade entre vetores
- **Norma L2** e sua conexão com normalização em transformers
- **Cosine similarity** como medida de similaridade semântica em embeddings
- **Scaled dot-product attention** a partir da definição matemática, sem bibliotecas

**Próximo:** `02_matrix_operations.ipynb` — como a multiplicação matricial generaliza o produto interno para computar atenção em paralelo sobre toda a sequência.